# 02 — Prétraitement et encodage

- Séparation `X` / `y`, **split stratifié 80/20** (`random_state=42`).
- Construction du `ColumnTransformer` défini dans `src/features.py` :
  numériques (imputation médiane + standardisation, `log1p` pour le salaire), catégorielles (imputation + One-Hot), `HaveWorkedWith` (encodage multi-hot des 40 technologies les plus fréquentes).
- Ajustement sur le **train uniquement** ; le test est seulement transformé (pas de fuite de données).

> En notebook 03 le préprocesseur est replacé *dans* le Pipeline de chaque modèle, afin que la validation croisée le ré-ajuste à chaque pli.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../src')
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from features import build_preprocessor, TARGET

df = pd.read_csv('../data/processed/applicants_clean.csv', keep_default_na=False, na_values=[''])
df['HaveWorkedWith'] = df['HaveWorkedWith'].fillna('')
df.shape

(73458, 14)

In [2]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print('Part de Employed=1  train :', round(y_train.mean(), 4), '| test :', round(y_test.mean(), 4))

(58766, 13) (14692, 13)
Part de Employed=1  train : 0.5362 | test : 0.5362


In [3]:
# Sauvegarde du split pour le notebook 03
X_train.assign(**{TARGET: y_train}).to_csv('../data/processed/train.csv', index=False)
X_test.assign(**{TARGET: y_test}).to_csv('../data/processed/test.csv', index=False)

In [4]:
prep = build_preprocessor(top_k_techs=40)
Xtr = prep.fit_transform(X_train)      # fit sur le train uniquement
Xte = prep.transform(X_test)           # transform seulement sur le test
print('Matrice train :', Xtr.shape, '| test :', Xte.shape, '| type :', type(Xtr).__name__)
names = prep.get_feature_names_out()
print(len(names), 'variables ->', list(names[:6]), '...')

Matrice train : (58766, 129) | test : (14692, 129) | type : csr_matrix
129 variables -> ['num__YearsCode', 'num__YearsCodePro', 'num__ComputerSkills', 'num__Employment', 'salary__PreviousSalary', 'cat__Age_<35'] ...


In [5]:
# Technologies retenues (multi-hot)
techs = prep.named_transformers_['tech'].get_feature_names_out()
print(len(techs), 'technologies :', list(techs))

40 technologies : ['ASP.NET', 'ASP.NET Core', 'AWS', 'Angular', 'Bash/Shell', 'C', 'C#', 'C++', 'DigitalOcean', 'Docker', 'Elasticsearch', 'Express', 'Firebase', 'Git', 'Google Cloud Platform', 'HTML/CSS', 'Heroku', 'Java', 'JavaScript', 'Kubernetes', 'MariaDB', 'Microsoft Azure', 'Microsoft SQL Server', 'MongoDB', 'MySQL', 'Node.js', 'Oracle', 'PHP', 'PostgreSQL', 'PowerShell', 'Python', 'React.js', 'Redis', 'SQL', 'SQLite', 'TypeScript', 'Vue.js', 'Yarn', 'jQuery', 'npm']


In [6]:
joblib.dump(prep, '../models/preprocessor.pkl')
print('Préprocesseur (ajusté sur le train) sauvegardé -> models/preprocessor.pkl')

Préprocesseur (ajusté sur le train) sauvegardé -> models/preprocessor.pkl
